# Vision Module Training Notebook V1

This notebook is written for **Google Colab** and is the starting point for the **vision module** of HDDS.

## Goal

Train a practical drone detector using the three datasets selected for the project:

1. **Anti-UAV300**
2. **Drone-vs-Bird Challenge**
3. **DroneSwarms**

## Main Training Strategy

This notebook follows a staged strategy instead of mixing everything blindly:

1. **Stage 1:** build a strong **drone-only baseline** using Anti-UAV300 and DroneSwarms.
2. **Stage 2:** fine-tune on **Drone-vs-Bird** so the model learns to avoid bird false alarms.
3. **Stage 3:** validate the final model on drone, bird, and hard-negative scenes.

## Recommended Model Family

Start with **YOLO11** in Colab because it is practical, strong, and easy to iterate.

Recommended first experiments:

- `yolo11n.pt` for fast debugging
- `yolo11s.pt` for the first serious baseline

## Important Scope Note

This notebook does **not** try to auto-download restricted datasets.

Some of your target datasets require:

- manual download,
- request approval,
- or signed usage agreements.

That is correct. The notebook will assume you already placed the raw datasets in Google Drive.


## Outline

1. Install dependencies in Colab.
2. Mount Google Drive.
3. Define the dataset layout expected by this notebook.
4. Prepare helper functions for YOLO-format conversion.
5. Convert Anti-UAV300 into YOLO format.
6. Convert Drone-vs-Bird into YOLO format.
7. Ingest DroneSwarms into YOLO format.
8. Build merged train/val datasets.
9. Train the Stage 1 drone detector.
10. Fine-tune the Stage 2 drone-vs-bird model.
11. Validate and export the trained weights.


## Step 0 - Colab Setup And Repository Cloning

This notebook now includes both approaches:

1. **recommended practical setup:** install the Ultralytics package directly,
2. **optional repo setup:** clone the YOLO/Ultralytics repository and the dataset-related repositories for reference scripts and metadata.

For training in Colab, the package install is usually enough. The repo clone is useful when you want to inspect configs, examples, or patch behavior.


In [ ]:
# Base Colab environment setup
!pip install -q ultralytics==8.3.71 opencv-python-headless==4.10.0.84 pyyaml tqdm pandas matplotlib gdown


In [ ]:
# Optional: clone useful repositories for reference and helper scripts
%cd /content
!git clone https://github.com/ultralytics/ultralytics.git
!git clone https://github.com/ZhaoJ9014/Anti-UAV.git external_anti_uav
!git clone https://github.com/wosdetc/challenge.git external_drone_vs_bird
!git clone https://github.com/Gradiant/pyodi.git external_pyodi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 1 - Dataset Acquisition Reality

Your three target datasets are **not all equally downloadable**.

### Anti-UAV300

This dataset is publicly referenced from the official Anti-UAV repository, but access is still typically via hosted file links.

### Drone-vs-Bird Challenge

This dataset requires a **data usage agreement (DUA)** and a request email.

Official request path:

- repository: `https://github.com/wosdetc/challenge`
- request email: `wosdetc@googlegroups.com`

### DroneSwarms

This dataset requires a **request form and approval**.

Official request path:

- website: `https://hiyuur.github.io/`
- contact shown there for application

### Practical consequence

This notebook can automate:

1. folder setup,
2. unzip/copy steps,
3. conversion,
4. training.

It **cannot legally bypass** DUA/request-based access. That part must remain manual.


## Step 2 - How To Bring The Datasets Into Colab

There are three valid workflows.

### Workflow A - Best for large datasets

1. download the dataset manually outside Colab,
2. upload the zip or extracted folder into Google Drive,
3. let this notebook extract/use it.

### Workflow B - If you already have a direct official download link

Use `gdown` with the official share link.

### Workflow C - If the dataset is restricted

Submit the request first, get the approved download link, then place it into Google Drive.

This notebook supports all three.


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/HDDS')
RAW_ROOT = DRIVE_ROOT / 'raw_datasets'
PREP_ROOT = DRIVE_ROOT / 'prepared_datasets'
RUNS_ROOT = DRIVE_ROOT / 'training_runs'
EXPORT_ROOT = DRIVE_ROOT / 'exports'
DOWNLOADS_ROOT = DRIVE_ROOT / 'downloads'

for folder in [RAW_ROOT, PREP_ROOT, RUNS_ROOT, EXPORT_ROOT, DOWNLOADS_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Downloads root:', DOWNLOADS_ROOT)
print('Raw datasets root:', RAW_ROOT)


## Step 3 - Official Download Link Placeholders

If you obtain official download URLs, paste them here.

Important:

- leave empty strings if you do not have the links yet,
- do not guess URLs,
- keep the official source only.


In [ ]:
ANTI_UAV300_URL = ''
DRONE_VS_BIRD_URL = ''
DRONESWARMS_URL = ''

ANTI_UAV300_ARCHIVE = DOWNLOADS_ROOT / 'Anti-UAV300.zip'
DRONE_VS_BIRD_ARCHIVE = DOWNLOADS_ROOT / 'Drone-vs-Bird.zip'
DRONESWARMS_ARCHIVE = DOWNLOADS_ROOT / 'DroneSwarms.zip'


In [ ]:
def maybe_gdown(url: str, output_path: Path):
    if not url:
        print(f'Skipping download for {output_path.name}: no URL provided')
        return
    output_path.parent.mkdir(parents=True, exist_ok=True)
    !gdown --fuzzy "$url" -O "$output_path"

# Uncomment when you have official direct links.
# maybe_gdown(ANTI_UAV300_URL, ANTI_UAV300_ARCHIVE)
# maybe_gdown(DRONE_VS_BIRD_URL, DRONE_VS_BIRD_ARCHIVE)
# maybe_gdown(DRONESWARMS_URL, DRONESWARMS_ARCHIVE)


## Step 4 - Manual Upload Shortcut

For small files, Colab upload works. For these datasets, it is usually better to upload to Drive outside Colab.

This cell is included only as a fallback.


In [ ]:
# Fallback only. Large datasets should be uploaded directly to Google Drive, not through this widget.
# from google.colab import files
# uploaded = files.upload()


## Step 5 - Extract Archives From Google Drive

If you already placed dataset archives inside `MyDrive/HDDS/downloads`, this cell will extract them into `raw_datasets/`.


In [ ]:
import shutil

ARCHIVE_MAP = {
    ANTI_UAV300_ARCHIVE: RAW_ROOT / 'Anti-UAV300',
    DRONE_VS_BIRD_ARCHIVE: RAW_ROOT / 'Drone-vs-Bird',
    DRONESWARMS_ARCHIVE: RAW_ROOT / 'DroneSwarms',
}

for archive_path, extract_dir in ARCHIVE_MAP.items():
    if archive_path.exists() and not extract_dir.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        print(f'Extracting {archive_path.name} -> {extract_dir}')
        shutil.unpack_archive(str(archive_path), str(extract_dir))
    else:
        print(f'Skipping {archive_path.name}: archive missing or folder already exists')


## Step 6 - Verify Raw Dataset Presence

Do this before running any conversion logic.


In [ ]:
ANTI_UAV_ROOT = RAW_ROOT / 'Anti-UAV300'
DVB_ROOT = RAW_ROOT / 'Drone-vs-Bird'
DRONESWARMS_ROOT = RAW_ROOT / 'DroneSwarms'

for name, path in {
    'Anti-UAV300': ANTI_UAV_ROOT,
    'Drone-vs-Bird': DVB_ROOT,
    'DroneSwarms': DRONESWARMS_ROOT,
}.items():
    print(name, 'exists ->', path.exists(), '| path ->', path)
    if path.exists():
        items = list(path.iterdir())[:10]
        print('  sample entries:', [item.name for item in items])


## Class Policy

The final false-alarm-aware vision model should use **two classes**:

1. `drone`
2. `bird`

However, not all source datasets contain both classes.

### Recommended use of the datasets

- **Anti-UAV300:** drone only
- **DroneSwarms:** drone only
- **Drone-vs-Bird:** drone and bird, plus hard negative sky/background scenes

### Training policy

- **Stage 1 dataset:** drone-only merged dataset
- **Stage 2 dataset:** two-class fine-tuning dataset

That is the cleanest way to get both sensitivity and bird rejection.


In [ ]:
CLASS_NAMES_STAGE1 = ['drone']
CLASS_NAMES_STAGE2 = ['drone', 'bird']

DVB_CLASS_MAP = {
    # Update these if your Drone-vs-Bird annotations use different class ids.
    # Example assumption:
    # 0 -> drone
    # 1 -> bird
    0: 'drone',
    1: 'bird',
}

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI'}


## Helper Functions

These helpers do three things:

1. convert bounding boxes to YOLO format,
2. create train/val folder structures,
3. copy images and write labels in a consistent way.


In [ ]:
def make_yolo_dirs(root: Path) -> None:
    for split in ['train', 'val']:
        (root / split / 'images').mkdir(parents=True, exist_ok=True)
        (root / split / 'labels').mkdir(parents=True, exist_ok=True)


def xywh_to_yolo(x: float, y: float, w: float, h: float, img_w: int, img_h: int):
    cx = (x + w / 2.0) / img_w
    cy = (y + h / 2.0) / img_h
    nw = w / img_w
    nh = h / img_h
    return cx, cy, nw, nh


def clamp_box(x: float, y: float, w: float, h: float, img_w: int, img_h: int):
    x = max(0.0, min(x, img_w - 1.0))
    y = max(0.0, min(y, img_h - 1.0))
    w = max(0.0, min(w, img_w - x))
    h = max(0.0, min(h, img_h - y))
    return x, y, w, h


def write_yolo_label(label_path: Path, boxes: list[tuple[int, float, float, float, float]]) -> None:
    lines = [f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for cls_id, cx, cy, w, h in boxes]
    label_path.write_text('
'.join(lines), encoding='utf-8')


def save_image_and_label(image_src: Path, image_dst: Path, label_dst: Path, boxes):
    image_dst.parent.mkdir(parents=True, exist_ok=True)
    label_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(image_src, image_dst)
    write_yolo_label(label_dst, boxes)


def save_yaml(dataset_root: Path, class_names: list[str], yaml_name: str = 'data.yaml') -> Path:
    payload = {
        'path': str(dataset_root),
        'train': 'train/images',
        'val': 'val/images',
        'names': {idx: name for idx, name in enumerate(class_names)},
    }
    yaml_path = dataset_root / yaml_name
    yaml_path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')
    return yaml_path


def split_sequence_names(names: list[str], train_ratio: float = 0.8):
    names = sorted(names)
    rng = random.Random(SEED)
    rng.shuffle(names)
    cutoff = max(1, int(len(names) * train_ratio))
    return set(names[:cutoff]), set(names[cutoff:])


## Anti-UAV300 Preparation

### What we know from the public benchmark structure

Anti-UAV sequences typically contain image frames and a label JSON such as `IR_label.json` or `RGB_label.json`, with fields like:

- `gt_rect`
- `exist`

The public challenge is mainly designed for tracking, so we convert it into **frame-based object detection data**.

### Practical choice for this notebook

For the first vision module, use **RGB frames only** unless you explicitly want an RGB+IR branch later.

### Assumption used below

The conversion code assumes sequence folders that contain:

- frame images or videos,
- and `*_label.json` files.

If your local folder structure differs, update the glob patterns in the conversion cell.


In [ ]:
def discover_anti_uav_sequences(root: Path) -> list[Path]:
    return sorted([path for path in root.iterdir() if path.is_dir()])


def find_sequence_images(seq_dir: Path) -> list[Path]:
    candidates = sorted([p for p in seq_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
    return candidates


def find_label_json(seq_dir: Path, prefer_rgb: bool = True) -> Path | None:
    names = ['RGB_label.json', 'IR_label.json'] if prefer_rgb else ['IR_label.json', 'RGB_label.json']
    for name in names:
        path = seq_dir / name
        if path.exists():
            return path
    json_files = sorted(seq_dir.glob('*_label.json'))
    return json_files[0] if json_files else None


def prepare_anti_uav300_yolo(output_root: Path, prefer_rgb: bool = True, train_ratio: float = 0.8):
    make_yolo_dirs(output_root)
    sequences = discover_anti_uav_sequences(ANTI_UAV_ROOT)
    train_seq, val_seq = split_sequence_names([p.name for p in sequences], train_ratio=train_ratio)

    counts = Counter()

    for seq_dir in tqdm(sequences, desc='Anti-UAV300'):
        label_json = find_label_json(seq_dir, prefer_rgb=prefer_rgb)
        images = find_sequence_images(seq_dir)

        if label_json is None or not images:
            print(f'Skipping {seq_dir.name}: missing images or label json')
            continue

        data = json.loads(label_json.read_text(encoding='utf-8'))
        gt_rect = data.get('gt_rect', [])
        exist = data.get('exist', [1] * len(gt_rect))

        split = 'train' if seq_dir.name in train_seq else 'val'

        for idx, image_path in enumerate(images):
            if idx >= len(gt_rect):
                break

            image = cv2.imread(str(image_path))
            if image is None:
                continue
            img_h, img_w = image.shape[:2]

            frame_boxes = []
            box = gt_rect[idx]
            is_present = exist[idx] if idx < len(exist) else 1

            if is_present and len(box) == 4 and sum(box) > 0:
                x, y, w, h = clamp_box(*box, img_w=img_w, img_h=img_h)
                if w > 1 and h > 1:
                    frame_boxes.append((0, *xywh_to_yolo(x, y, w, h, img_w, img_h)))
                    counts['objects'] += 1

            out_name = f"antiuav_{seq_dir.name}_{idx:06d}{image_path.suffix.lower()}"
            image_dst = output_root / split / 'images' / out_name
            label_dst = output_root / split / 'labels' / (Path(out_name).stem + '.txt')
            save_image_and_label(image_path, image_dst, label_dst, frame_boxes)
            counts[f'{split}_images'] += 1

    yaml_path = save_yaml(output_root, CLASS_NAMES_STAGE1)
    print('Anti-UAV300 YOLO dataset ready at:', output_root)
    print('YAML:', yaml_path)
    print(dict(counts))
    return yaml_path


## Drone-vs-Bird Preparation

### Why this dataset is special

This dataset is the key false-alarm dataset because it explicitly stresses the **drone vs bird** confusion.

### Public annotation description

The official repository states that annotations are stored as text files with one line per frame, containing:

```text
framenum num_objs obj1_x obj1_y obj1_w obj1_h obj1_class ...
```

### Important warning

The exact class-id mapping may differ across challenge editions.

Before training, inspect one annotation file and confirm whether:

- `0` means drone and `1` means bird,
- or the reverse,
- or a different mapping is used.

The notebook exposes `DVB_CLASS_MAP` for this reason.


In [ ]:
def read_dvb_annotation_file(annotation_path: Path):
    records = {}
    with annotation_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            parts = line.strip().split()
            if not parts:
                continue
            frame_num = int(parts[0])
            num_objs = int(parts[1])
            objects = []
            cursor = 2
            for _ in range(num_objs):
                x = float(parts[cursor]); y = float(parts[cursor + 1])
                w = float(parts[cursor + 2]); h = float(parts[cursor + 3])
                cls_id = int(parts[cursor + 4])
                objects.append((x, y, w, h, cls_id))
                cursor += 5
            records[frame_num] = objects
    return records


def inspect_dvb_classes(annotation_files: list[Path]):
    seen = Counter()
    for file_path in annotation_files:
        records = read_dvb_annotation_file(file_path)
        for objects in records.values():
            for *_, cls_id in objects:
                seen[cls_id] += 1
    print('Observed raw class ids:', dict(seen))


def prepare_dvb_yolo(output_root: Path, train_ratio: float = 0.8):
    make_yolo_dirs(output_root)

    annotation_files = sorted(DVB_ROOT.rglob('*.txt'))
    if not annotation_files:
        raise FileNotFoundError('No Drone-vs-Bird annotation .txt files found under DVB_ROOT')

    inspect_dvb_classes(annotation_files[: min(10, len(annotation_files))])

    sequence_names = [p.stem for p in annotation_files]
    train_seq, val_seq = split_sequence_names(sequence_names, train_ratio=train_ratio)
    counts = Counter()

    for ann_path in tqdm(annotation_files, desc='Drone-vs-Bird'):
        records = read_dvb_annotation_file(ann_path)
        seq_name = ann_path.stem
        split = 'train' if seq_name in train_seq else 'val'

        frame_dirs = [ann_path.parent / seq_name, ann_path.parent / 'frames' / seq_name]
        frame_dir = next((p for p in frame_dirs if p.exists()), None)
        if frame_dir is None:
            print(f'Skipping {seq_name}: frame directory not found')
            continue

        image_files = sorted([p for p in frame_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
        if not image_files:
            continue

        for image_path in image_files:
            try:
                frame_num = int(image_path.stem)
            except ValueError:
                continue

            image = cv2.imread(str(image_path))
            if image is None:
                continue
            img_h, img_w = image.shape[:2]

            frame_boxes = []
            for x, y, w, h, raw_cls in records.get(frame_num, []):
                class_name = DVB_CLASS_MAP.get(raw_cls)
                if class_name is None:
                    continue
                cls_id = CLASS_NAMES_STAGE2.index(class_name)
                x, y, w, h = clamp_box(x, y, w, h, img_w, img_h)
                if w > 1 and h > 1:
                    frame_boxes.append((cls_id, *xywh_to_yolo(x, y, w, h, img_w, img_h)))
                    counts[class_name] += 1

            out_name = f"dvb_{seq_name}_{frame_num:06d}{image_path.suffix.lower()}"
            image_dst = output_root / split / 'images' / out_name
            label_dst = output_root / split / 'labels' / (Path(out_name).stem + '.txt')
            save_image_and_label(image_path, image_dst, label_dst, frame_boxes)
            counts[f'{split}_images'] += 1

    yaml_path = save_yaml(output_root, CLASS_NAMES_STAGE2)
    print('Drone-vs-Bird YOLO dataset ready at:', output_root)
    print('YAML:', yaml_path)
    print(dict(counts))
    return yaml_path


## DroneSwarms Preparation

DroneSwarms is mainly useful for very small drone instances.

Because public access is controlled and packaging may change, this notebook supports two practical cases:

1. the dataset is already in **YOLO format**,
2. or it is in **COCO JSON format** and needs conversion.

If your package is different, use the inspection cells first and then adjust the loader.


In [ ]:
def find_coco_json(root: Path) -> Path | None:
    candidates = list(root.rglob('*.json'))
    return candidates[0] if candidates else None


def ingest_existing_yolo_dataset(src_root: Path, dst_root: Path, class_names: list[str]):
    make_yolo_dirs(dst_root)
    for split in ['train', 'val']:
        src_images = src_root / split / 'images'
        src_labels = src_root / split / 'labels'
        if not src_images.exists() or not src_labels.exists():
            continue
        for image_path in tqdm(sorted(src_images.iterdir()), desc=f'Copy {split}'):
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue
            label_path = src_labels / (image_path.stem + '.txt')
            out_image = dst_root / split / 'images' / image_path.name
            out_label = dst_root / split / 'labels' / label_path.name
            shutil.copy2(image_path, out_image)
            if label_path.exists():
                shutil.copy2(label_path, out_label)
            else:
                out_label.write_text('', encoding='utf-8')
    return save_yaml(dst_root, class_names)


def convert_coco_to_yolo(coco_json_path: Path, image_root: Path, dst_root: Path, class_name_map: dict[int, str], train_ratio: float = 0.8):
    make_yolo_dirs(dst_root)
    data = json.loads(coco_json_path.read_text(encoding='utf-8'))

    images = {item['id']: item for item in data['images']}
    anns_by_image = defaultdict(list)
    for ann in data['annotations']:
        anns_by_image[ann['image_id']].append(ann)

    image_ids = sorted(images)
    rng = random.Random(SEED)
    rng.shuffle(image_ids)
    cutoff = max(1, int(len(image_ids) * train_ratio))
    train_ids = set(image_ids[:cutoff])

    for image_id, image_info in tqdm(images.items(), desc='DroneSwarms COCO'):
        split = 'train' if image_id in train_ids else 'val'
        image_path = image_root / image_info['file_name']
        if not image_path.exists():
            continue
        img_w = int(image_info['width'])
        img_h = int(image_info['height'])
        boxes = []
        for ann in anns_by_image.get(image_id, []):
            class_name = class_name_map.get(int(ann['category_id']))
            if class_name is None:
                continue
            cls_id = CLASS_NAMES_STAGE1.index(class_name)
            x, y, w, h = clamp_box(*ann['bbox'], img_w=img_w, img_h=img_h)
            if w > 1 and h > 1:
                boxes.append((cls_id, *xywh_to_yolo(x, y, w, h, img_w, img_h)))
        out_image = dst_root / split / 'images' / image_path.name
        out_label = dst_root / split / 'labels' / f'{image_path.stem}.txt'
        save_image_and_label(image_path, out_image, out_label, boxes)

    return save_yaml(dst_root, CLASS_NAMES_STAGE1)


def prepare_droneswarms_yolo(output_root: Path):
    yolo_yaml = DRONESWARMS_ROOT / 'data.yaml'
    if yolo_yaml.exists():
        print('Detected existing YOLO-format DroneSwarms dataset.')
        return ingest_existing_yolo_dataset(DRONESWARMS_ROOT, output_root, CLASS_NAMES_STAGE1)

    coco_json = find_coco_json(DRONESWARMS_ROOT)
    if coco_json is not None:
        print('Detected COCO-format DroneSwarms dataset:', coco_json)
        image_root = coco_json.parent if (coco_json.parent / 'images').exists() else DRONESWARMS_ROOT
        class_name_map = {1: 'drone'}
        return convert_coco_to_yolo(coco_json, image_root, output_root, class_name_map)

    raise FileNotFoundError('Could not detect YOLO or COCO annotations for DroneSwarms. Inspect the raw package and update this loader.')


## Build Prepared Datasets

This notebook will create three prepared datasets:

1. `anti_uav_yolo`
2. `dvb_yolo`
3. `droneswarms_yolo`

Then it will create two merged datasets:

1. `merged_stage1_drone_only`
2. `merged_stage2_drone_bird`


In [ ]:
ANTI_UAV_YOLO = PREP_ROOT / 'anti_uav_yolo'
DVB_YOLO = PREP_ROOT / 'dvb_yolo'
DRONESWARMS_YOLO = PREP_ROOT / 'droneswarms_yolo'
MERGED_STAGE1 = PREP_ROOT / 'merged_stage1_drone_only'
MERGED_STAGE2 = PREP_ROOT / 'merged_stage2_drone_bird'


In [ ]:
# Run these one by one after your raw datasets are in place.
# Uncomment when ready.

# anti_uav_yaml = prepare_anti_uav300_yolo(ANTI_UAV_YOLO)
# dvb_yaml = prepare_dvb_yolo(DVB_YOLO)
# droneswarms_yaml = prepare_droneswarms_yolo(DRONESWARMS_YOLO)


## Merge Helper

Stage 1 should merge **drone-only** data.

Stage 2 should merge:

- Stage 1 drone images,
- plus Drone-vs-Bird data,
- with a final two-class label space.


In [ ]:
def merge_yolo_datasets(sources: list[Path], dst_root: Path, class_names: list[str]):
    if dst_root.exists():
        shutil.rmtree(dst_root)
    make_yolo_dirs(dst_root)

    counts = Counter()
    for source in sources:
        for split in ['train', 'val']:
            for image_path in sorted((source / split / 'images').glob('*')):
                if image_path.suffix.lower() not in IMAGE_EXTS:
                    continue
                label_path = source / split / 'labels' / f'{image_path.stem}.txt'
                out_image = dst_root / split / 'images' / f'{source.name}_{image_path.name}'
                out_label = dst_root / split / 'labels' / f'{source.name}_{image_path.stem}.txt'
                shutil.copy2(image_path, out_image)
                if label_path.exists():
                    shutil.copy2(label_path, out_label)
                else:
                    out_label.write_text('', encoding='utf-8')
                counts[f'{split}_images'] += 1

    yaml_path = save_yaml(dst_root, class_names)
    print('Merged dataset ready at:', dst_root)
    print('YAML:', yaml_path)
    print(dict(counts))
    return yaml_path


In [ ]:
# Stage 1: drone-only baseline
# stage1_yaml = merge_yolo_datasets(
#     sources=[ANTI_UAV_YOLO, DRONESWARMS_YOLO],
#     dst_root=MERGED_STAGE1,
#     class_names=CLASS_NAMES_STAGE1,
# )

# Stage 2: fine-tuning dataset with bird rejection
# stage2_yaml = merge_yolo_datasets(
#     sources=[MERGED_STAGE1, DVB_YOLO],
#     dst_root=MERGED_STAGE2,
#     class_names=CLASS_NAMES_STAGE2,
# )


## Stage 1 Training: Drone-Only Baseline

Recommended first serious run:

- model: `yolo11s.pt`
- image size: `1280` if Colab GPU memory allows
- epochs: `50` to `100`
- batch: auto or conservative

If memory is tight, start with:

- `imgsz=960`
- `batch=8`


In [ ]:
# Uncomment after stage1_yaml exists.

# model_stage1 = YOLO('yolo11s.pt')
# results_stage1 = model_stage1.train(
#     data=str(stage1_yaml),
#     epochs=60,
#     imgsz=1280,
#     batch=8,
#     project=str(RUNS_ROOT),
#     name='vision_stage1_drone_only',
#     seed=SEED,
#     workers=2,
#     cache=False,
# )


## Stage 2 Fine-Tuning: Drone vs Bird

This is the most important stage for false-alarm control.

Recommended logic:

1. start from the best Stage 1 checkpoint,
2. fine-tune on the two-class merged dataset,
3. evaluate whether bird false detections are reduced without destroying drone recall.


In [ ]:
# Uncomment after stage2_yaml exists and Stage 1 training is complete.

# stage1_best = RUNS_ROOT / 'vision_stage1_drone_only' / 'weights' / 'best.pt'
# model_stage2 = YOLO(str(stage1_best))
# results_stage2 = model_stage2.train(
#     data=str(stage2_yaml),
#     epochs=40,
#     imgsz=1280,
#     batch=8,
#     project=str(RUNS_ROOT),
#     name='vision_stage2_drone_vs_bird',
#     seed=SEED,
#     workers=2,
#     cache=False,
# )


## Validation

Run validation after each stage.

For the final model, focus on:

1. `mAP50`
2. `mAP50-95`
3. drone recall
4. bird confusion behavior
5. failure cases on tiny targets


In [ ]:
# Uncomment after Stage 2 training.

# final_weights = RUNS_ROOT / 'vision_stage2_drone_vs_bird' / 'weights' / 'best.pt'
# final_model = YOLO(str(final_weights))
# final_metrics = final_model.val(data=str(stage2_yaml), imgsz=1280)
# print(final_metrics)


## Quick Inference Check

Use this cell to visually inspect a few images after training.


In [ ]:
# sample_images = list((MERGED_STAGE2 / 'val' / 'images').glob('*'))[:8]
# preds = final_model.predict([str(p) for p in sample_images], conf=0.25, imgsz=1280, save=True)


## Export Best Weights

After training, copy the best checkpoint into a stable export folder.


In [ ]:
# exported_weight = EXPORT_ROOT / 'vision_yolo11_stage2_best.pt'
# shutil.copy2(final_weights, exported_weight)
# print('Exported to:', exported_weight)


## What You Should Tune First

Tune these in order:

1. dataset conversion correctness,
2. class mapping correctness,
3. image size,
4. confidence thresholds,
5. training epochs,
6. bird false positives,
7. tiny-drone recall.

Do **not** start by tuning dozens of hyperparameters before verifying that the prepared datasets are correct.


## Recommended Experiment Sequence

### Experiment A

- `yolo11n.pt`
- Anti-UAV300 only
- verify pipeline end to end

### Experiment B

- `yolo11s.pt`
- Anti-UAV300 + DroneSwarms
- drone-only baseline

### Experiment C

- Stage 1 best checkpoint
- fine-tune on Drone-vs-Bird
- evaluate bird false alarms

### Experiment D

- compare `yolo11n` vs `yolo11s`
- compare `imgsz=960` vs `1280`
- compare Stage 1 vs Stage 2 false alarms


## Common Failure Modes

1. **Wrong class-id mapping in Drone-vs-Bird**  
   This will silently poison the model.

2. **Dataset split leakage by sequence**  
   Do not split frames randomly if many frames come from the same video.

3. **Broken YOLO labels after conversion**  
   Always inspect random images with labels visually.

4. **Tiny drones disappearing at low image size**  
   Use larger `imgsz` where possible.

5. **Bird rejection improving while drone recall collapses**  
   This is exactly why Stage 1 and Stage 2 should be compared separately.


## Next Step After This Notebook

Once you finish the first serious training run, the next engineering step is:

1. export the best detector,
2. create a standalone vision inference module in `Project v1`,
3. add tracking,
4. then expose time-window confidence for fusion with radar and audio.
